In [ ]:
# Example 1: Public API surface — verify the documented names exist on the package.
import ssl
import truststore

expected = ["SSLContext", "inject_into_ssl", "extract_from_ssl"]
for name in expected:
    assert hasattr(truststore, name), f"missing public name: {name}"
    print(f"truststore.{name} -> {getattr(truststore, name)!r}")

# SSLContext should be a subclass of ssl.SSLContext
assert issubclass(truststore.SSLContext, ssl.SSLContext)
print("\ntruststore.SSLContext is a subclass of ssl.SSLContext")

In [ ]:
# Example 2: Construct a truststore SSLContext and inspect its basic attributes.
import ssl
import truststore

ctx = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
print(f"type(ctx) = {type(ctx).__name__}")
print(f"ctx.protocol = {ctx.protocol!r}")
print(f"ctx.check_hostname = {ctx.check_hostname}")
print(f"ctx.verify_mode = {ctx.verify_mode!r}")

# Defaults for a TLS_CLIENT context: hostname checking on, CERT_REQUIRED.
assert ctx.check_hostname is True
assert ctx.verify_mode == ssl.CERT_REQUIRED
assert isinstance(ctx, ssl.SSLContext)
print("\nSSLContext constructed with expected defaults")

In [ ]:
# Example 3: inject_into_ssl / extract_from_ssl round-trip.
# After inject_into_ssl(), ssl.SSLContext should resolve to truststore.SSLContext.
import ssl
import truststore

original = ssl.SSLContext
try:
    truststore.inject_into_ssl()
    assert ssl.SSLContext is truststore.SSLContext, "inject_into_ssl did not patch ssl.SSLContext"
    injected_ctx = ssl.create_default_context()
    print(f"after inject: type(ssl.create_default_context()) = {type(injected_ctx).__name__}")
    assert isinstance(injected_ctx, truststore.SSLContext)
finally:
    truststore.extract_from_ssl()

assert ssl.SSLContext is original, "extract_from_ssl did not restore ssl.SSLContext"
print(f"after extract: ssl.SSLContext restored to {ssl.SSLContext.__name__}")

In [ ]:
# Example 4: Negative test — connecting to a host with an invalid cert raises SSLCertVerificationError.
import ssl
import urllib.error
import urllib.request

import truststore

ctx = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
raised = None
try:
    with urllib.request.urlopen("https://self-signed.badssl.com/", context=ctx, timeout=10) as resp:
        resp.read(10)
except urllib.error.URLError as e:
    raised = e
    print(f"got expected URLError: {e.reason!r}")
except OSError as e:
    raised = e
    print(f"network unavailable: {type(e).__name__}: {e}")

# If we had network access, the bad-ssl host MUST have raised; if not, we still pass.
if raised is not None and isinstance(raised, urllib.error.URLError):
    reason = raised.reason
    assert isinstance(reason, ssl.SSLError) or "certificate" in str(reason).lower(), \
        f"unexpected non-cert failure: {reason!r}"
    print("\nbad-cert host correctly rejected by truststore SSLContext")
else:
    print("\nskipped live bad-cert assertion (no network)")

assert True